In [1]:
# ============================================================================
# 05_sin_industry_analysis.ipynb
# US sin-industry (S) proxy analysis added in revision.
# Collects SIC codes from the EDGAR submissions API, classifies sin industries
# (core: alcohol/tobacco/gaming; broad: + defense), and estimates the
# sin-minus-non-sin hedge under CAPM / FF3 / Carhart.
# ============================================================================

In [2]:
# --- Imports --------------------------------------------------------------
import urllib.request, json, time, os
import pandas as pd, numpy as np
import statsmodels.api as sm
HDR = {"User-Agent": "Anonymous Research anonymous@example.com"}
DATA_DIR, TAB_DIR = "../data", "../results/tables"

In [3]:
# --- Collect SIC codes from EDGAR submissions API (incremental) -----------
acc = pd.read_csv(f"{DATA_DIR}/accounting_panel.csv")
ciks = sorted(acc["cik"].unique())
out = f"{DATA_DIR}/sic_codes.csv"
done = {}
if os.path.exists(out):
    prev = pd.read_csv(out); done = dict(zip(prev["cik"], prev["sic"]))
for cik in [c for c in ciks if c not in done]:
    url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
    try:
        d = json.loads(urllib.request.urlopen(
            urllib.request.Request(url, headers=HDR), timeout=20).read())
        done[cik] = d.get("sic", "")
    except Exception:
        done[cik] = ""
    time.sleep(0.11)
pd.DataFrame([(k,v) for k,v in done.items()],
             columns=["cik","sic"]).to_csv(out, index=False)
print("SIC codes:", len(done))

SIC codes: 4661


In [4]:
# --- Classify sin industries by SIC ---------------------------------------
sic = pd.read_csv(f"{DATA_DIR}/sic_codes.csv")
sic["sic"] = pd.to_numeric(sic["sic"], errors="coerce")
acc = acc.merge(sic, on="cik", how="left")

def classify(s):
    if pd.isna(s): return "none"
    s = int(s)
    if 2080 <= s <= 2085 or 5180 <= s <= 5182: return "alcohol"
    if 2100 <= s <= 2199: return "tobacco"
    if s in (7990,7993,7996,7997,7999): return "gaming"
    if s in (3480,3482,3483,3484,3489,3760,3761,3764,3769,3795,3812): return "defense"
    return "none"

acc["sin_type"] = acc["sic"].apply(classify)
acc["sin_core"]  = acc["sin_type"].isin(["alcohol","tobacco","gaming"]).astype(int)
acc["sin_broad"] = (acc["sin_type"] != "none").astype(int)
acc.to_csv(f"{DATA_DIR}/accounting_panel_sic.csv", index=False)
print(acc["sin_type"].value_counts().to_string())

sin_type
none       28829
alcohol      138
defense      125
gaming       108
tobacco       41


In [5]:
# --- Estimate the sin-minus-non-sin hedge alphas --------------------------
# Prices for the full sin universe are in sin_prices.csv (collected separately
# because sin stocks fall outside the capped main universe).
px    = pd.read_csv(f"{DATA_DIR}/monthly_prices.csv", index_col=0, parse_dates=True)
sinpx = pd.read_csv(f"{DATA_DIR}/sin_prices.csv", index_col=0, parse_dates=True)
fac   = pd.read_csv(f"{DATA_DIR}/ff_factors.csv", index_col=0, parse_dates=True)
fac.index = fac.index.to_period("M").to_timestamp()

def to_rets(p):
    r = p.pct_change().iloc[1:]; r.index = r.index.to_period("M").to_timestamp()
    return r.clip(-0.9, 4.0)
rmain, rsin = to_rets(px), to_rets(sinpx)

sin_broad = acc.groupby("ticker")["sin_broad"].max()
sin_core  = acc.groupby("ticker")["sin_core"].max()
non_tks = [t for t in rmain.columns if sin_broad.get(t,0)==0]
r_non = rmain[non_tks].mean(axis=1)

def alphas(hedge):
    d = pd.DataFrame({"h":hedge}).join(fac, how="inner").dropna()
    rows=[]
    for name,cols in {"CAPM":["MktRF"],"FF3":["MktRF","SMB","HML"],
                      "Carhart4":["MktRF","SMB","HML","WML"]}.items():
        m = sm.OLS(d["h"], sm.add_constant(d[cols])).fit(
            cov_type="HAC", cov_kwds={"maxlags":3})
        rows.append({"model":name,"alpha_pct":round(m.params["const"]*100,3),
                     "t":round(m.tvalues["const"],2)})
    return pd.DataFrame(rows)

for label, flag in [("core",sin_core),("broad",sin_broad)]:
    tks = [t for t in rsin.columns if (flag.get(t,0)==1 or label=="broad")]
    r_sin = rsin[tks].mean(axis=1)
    tbl = alphas((r_sin - r_non).dropna())
    tbl.to_csv(f"{TAB_DIR}/us_sin_{label}_alphas.csv", index=False)
    print(f"--- {label} ---"); print(tbl.to_string(index=False))

--- core ---
   model  alpha_pct    t
    CAPM      2.992 4.74
     FF3      2.863 4.37
Carhart4      2.886 4.23
--- broad ---
   model  alpha_pct    t
    CAPM      2.019 4.56
     FF3      1.870 4.18
Carhart4      1.881 4.07
